In [3]:
# =============================================================================
# LDASN: Lightweight Dual-Attention Selective Network
# Cotton Leaf Curl Disease Detection
# Full Pipeline: Model + Training + All Metrics + XAI + 5-Fold CV
# =============================================================================

import os, time, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from PIL import Image
from tqdm import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler
from torchvision import transforms
from torchvision.datasets import ImageFolder

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef
)
from sklearn.calibration import calibration_curve

# Optional: fvcore for FLOPs (pip install fvcore)
try:
    from fvcore.nn import FlopCountAnalysis
    FVCORE_AVAILABLE = True
except ImportError:
    FVCORE_AVAILABLE = False

# Optional: pytorch-grad-cam (pip install grad-cam)
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    GRADCAM_AVAILABLE = True
except ImportError:
    GRADCAM_AVAILABLE = False

# =============================================================================
# CONFIG
# =============================================================================
CFG = {
    'data_dir'       : 'dataset',        # root folder with class subfolders
    'output_dir'     : 'results',
    'img_size'       : 224,
    'batch_size'     : 16,
    'epochs'         : 50,
    'lr'             : 3e-4,
    'weight_decay'   : 1e-4,
    'num_workers'    : 2,
    'seed'           : 42,
    'n_folds'        : 5,
    'patience'       : 10,              # early stopping
    'top_k_patches'  : 50,              # sparse patch selection
    'patch_size'     : 16,
    'd_model'        : 256,
    'n_heads'        : 4,
    'n_layers'       : 4,
    'conf_threshold' : 0.9,             # high-confidence threshold
    'device'         : 'cuda' if torch.cuda.is_available() else 'cpu',
}

os.makedirs(CFG['output_dir'], exist_ok=True)
torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])


# =============================================================================
# DATASET
# =============================================================================
def get_transforms(train=True):
    if train:
        return transforms.Compose([
            transforms.Resize((CFG['img_size'], CFG['img_size'])),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(20),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
    return transforms.Compose([
        transforms.Resize((CFG['img_size'], CFG['img_size'])),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])

def get_raw_transform():
    """No-norm transform for XAI visualization."""
    return transforms.Compose([
        transforms.Resize((CFG['img_size'], CFG['img_size'])),
        transforms.ToTensor(),
    ])


# =============================================================================
# MODEL COMPONENTS
# =============================================================================

# ---- Component 1: Depthwise Separable Multi-Scale Extractor ----
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, padding):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, kernel_size, padding=padding, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.bn(self.pw(self.dw(x))))


class SEBlock(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(ch, max(ch // r, 4)),
            nn.ReLU(),
            nn.Linear(max(ch // r, 4), ch),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.fc(x).view(x.size(0), -1, 1, 1)


class MultiScaleExtractor(nn.Module):
    """Two-scale depthwise separable + merged SE attention."""
    def __init__(self, in_ch=3, out_ch=128):
        super().__init__()
        mid = out_ch // 2
        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.GELU()
        )
        # Scale 1: fine texture (3x3)
        self.scale1 = nn.Sequential(
            DepthwiseSeparableConv(32, 64, 3, 1),
            DepthwiseSeparableConv(64, mid, 3, 1),
        )
        # Scale 2: broader curl patterns (5x5)
        self.scale2 = nn.Sequential(
            DepthwiseSeparableConv(32, 64, 5, 2),
            DepthwiseSeparableConv(64, mid, 5, 2),
        )
        # Merge + SE
        self.merge_se = SEBlock(out_ch)
        self.proj = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU()
        )
        # Shallow residual for patch tokenizer
        self.shallow = nn.Sequential(
            DepthwiseSeparableConv(32, out_ch, 1, 0),
        )

    def forward(self, x):
        s = self.stem(x)                        # B,32,H/2,W/2
        s1 = self.scale1(s)                     # B,64,H/2,W/2
        s2 = self.scale2(s)                     # B,64,H/2,W/2
        merged = torch.cat([s1, s2], dim=1)     # B,128,H/2,W/2
        merged = self.merge_se(merged)
        deep = self.proj(merged)                # B,128,H/2,W/2
        shallow = self.shallow(s)               # B,128,H/2,W/2  (for residual tokens)
        return deep, shallow                    # both: B,C,H',W'


# ---- Component 2: Sparse Patch Selector ----
class SparsePatchSelector(nn.Module):
    """
    Scores spatial positions by saliency, selects top-K patches.
    Returns patch tokens with residual shallow features added.
    """
    def __init__(self, in_ch, d_model, patch_size, top_k):
        super().__init__()
        self.patch_size = patch_size
        self.top_k = top_k
        # lightweight saliency head
        self.saliency = nn.Conv2d(in_ch, 1, 1)
        # patch projection
        self.proj = nn.Linear(in_ch * patch_size * patch_size, d_model)
        # positional embedding (learned)
        n_patches = (CFG['img_size'] // 2 // patch_size) ** 2
        self.pos_emb = nn.Embedding(n_patches, d_model)
        self.register_buffer('pos_ids', torch.arange(n_patches))

    def forward(self, deep, shallow):
        B, C, H, W = deep.shape
        P = self.patch_size
        # saliency map
        sal = self.saliency(deep)               # B,1,H,W
        # average-pool to patch grid
        n_h, n_w = H // P, W // P
        sal_patches = F.avg_pool2d(sal, P)      # B,1,n_h,n_w
        sal_flat = sal_patches.view(B, -1)      # B, N

        # select top-K indices per sample
        k = min(self.top_k, sal_flat.shape[1])
        _, top_idx = sal_flat.topk(k, dim=1)   # B, K

        # extract patch tokens
        feat = deep + shallow                   # residual merge
        # unfold into patches: B, C, n_h, n_w, P, P
        feat_unf = feat.unfold(2, P, P).unfold(3, P, P)
        feat_unf = feat_unf.contiguous().view(B, C, n_h * n_w, P * P)
        feat_unf = feat_unf.permute(0, 2, 1, 3).contiguous()
        feat_unf = feat_unf.view(B, n_h * n_w, C * P * P)  # B, N, C*P*P

        # gather top-K
        idx_exp = top_idx.unsqueeze(-1).expand(-1, -1, feat_unf.shape[-1])
        tokens = torch.gather(feat_unf, 1, idx_exp)        # B, K, C*P*P

        tokens = self.proj(tokens)             # B, K, d_model

        # positional embeddings for selected positions
        pos_e = self.pos_emb(self.pos_ids)     # N, d_model
        pos_e = pos_e.unsqueeze(0).expand(B, -1, -1)      # B, N, d_model
        idx_pos = top_idx.unsqueeze(-1).expand(-1, -1, CFG['d_model'])
        sel_pos = torch.gather(pos_e, 1, idx_pos)          # B, K, d_model

        tokens = tokens + sel_pos

        # also return channel-wise attention weights for Component 4
        ch_attn = self.saliency(deep).sigmoid().mean(dim=[2, 3])  # B,1
        return tokens, top_idx, ch_attn


# ---- Component 3: Lightweight Transformer ----
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, mlp_ratio=2.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        mlp_dim = int(d_model * mlp_ratio)
        self.mlp  = nn.Sequential(
            nn.Linear(d_model, mlp_dim), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, d_model), nn.Dropout(dropout),
        )

    def forward(self, x, ch_attn_mask=None):
        # ch_attn_mask: B,1 scalar — scales residual (Component 4 one-way feedback)
        norm_x = self.norm1(x)
        attn_out, _ = self.attn(norm_x, norm_x, norm_x)
        if ch_attn_mask is not None:
            attn_out = attn_out * ch_attn_mask.unsqueeze(-1)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class TinyTransformer(nn.Module):
    def __init__(self, d_model, n_heads, n_layers):
        super().__init__()
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, tokens, ch_attn):
        B = tokens.shape[0]
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, tokens], dim=1)
        for blk in self.blocks:
            x = blk(x, ch_attn)
        x = self.norm(x)
        return x[:, 0]   # CLS token


# ---- Component 5: Temperature Scaling Head ----
class TemperatureScaledHead(nn.Module):
    def __init__(self, d_model, num_classes):
        super().__init__()
        self.fc = nn.Linear(d_model, num_classes)
        self.temperature = nn.Parameter(torch.ones(1))

    def forward(self, x):
        logits = self.fc(x)
        return logits / self.temperature.clamp(min=0.1)


# ---- Full LDASN Model ----
class LDASN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.num_classes = num_classes
        C = 128
        self.extractor  = MultiScaleExtractor(in_ch=3, out_ch=C)
        self.selector   = SparsePatchSelector(
            in_ch=C, d_model=CFG['d_model'],
            patch_size=CFG['patch_size'], top_k=CFG['top_k_patches']
        )
        self.transformer = TinyTransformer(CFG['d_model'], CFG['n_heads'], CFG['n_layers'])
        self.head        = TemperatureScaledHead(CFG['d_model'], num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                # Conv2d weights are 4D — kaiming is safe
                nn.init.kaiming_normal_(m.weight, mode='fan_out',
                                        nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                # BN weight is 1D — use constant init, not kaiming
                if m.weight is not None:
                    nn.init.ones_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        deep, shallow = self.extractor(x)
        tokens, top_idx, ch_attn = self.selector(deep, shallow)
        cls_feat = self.transformer(tokens, ch_attn)
        logits = self.head(cls_feat)
        return logits

    def get_cam_target_layer(self):
        """Return layer for Grad-CAM."""
        return self.extractor.proj[-2]  # last BN before GELU in proj


# =============================================================================
# TRAINING UTILITIES
# =============================================================================
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def compute_flops(model, device):
    if not FVCORE_AVAILABLE:
        return "fvcore not installed"
    model.eval()
    dummy = torch.randn(1, 3, CFG['img_size'], CFG['img_size']).to(device)
    try:
        flops = FlopCountAnalysis(model, dummy)
        return flops.total()
    except Exception as e:
        return f"FLOPs error: {e}"


def measure_inference_time(model, device, n=100):
    model.eval()
    dummy = torch.randn(1, 3, CFG['img_size'], CFG['img_size']).to(device)
    # warmup
    with torch.no_grad():
        for _ in range(10): model(dummy)
    if device == 'cuda':
        torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        for _ in range(n): model(dummy)
    if device == 'cuda':
        torch.cuda.synchronize()
    return (time.time() - start) / n * 1000  # ms


# =============================================================================
# FOCAL LOSS  (domain standard for imbalanced multi-class disease datasets)
# =============================================================================
class FocalLoss(nn.Module):
    """
    Focal Loss: FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    gamma > 0 reduces relative loss for well-classified examples,
    focusing training on hard / rare classes (e.g. Leaf Variegation).
    alpha: per-class weights (inverse frequency), same as weighted CE.
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma     = gamma
        self.reduction = reduction
        # alpha will be set after dataset is loaded
        self.register_buffer('alpha', alpha if alpha is not None
                             else torch.ones(1))

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, reduction='none',
                               weight=self.alpha.to(logits.device))
        pt   = torch.exp(-ce)                      # probability of correct class
        loss = (1 - pt) ** self.gamma * ce
        return loss.mean() if self.reduction == 'mean' else loss.sum()


def get_class_weights(dataset):
    """Inverse-frequency weights, one per class."""
    targets = np.array(dataset.targets)
    counts  = np.bincount(targets)
    weights = 1.0 / counts.astype(float)
    weights = weights / weights.sum() * len(counts)   # normalise so mean=1
    return torch.tensor(weights, dtype=torch.float32)


def get_sample_weights(dataset):
    """Per-sample weights for WeightedRandomSampler."""
    targets      = np.array(dataset.targets)
    class_counts = np.bincount(targets)
    class_w      = 1.0 / class_counts.astype(float)
    sample_w     = class_w[targets]
    return sample_w


# =============================================================================
class EarlyStopping:
    def __init__(self, patience, path):
        self.patience = patience
        self.best = -np.inf
        self.counter = 0
        self.path = path

    def step(self, val_acc, model):
        if val_acc > self.best:
            self.best = val_acc
            torch.save(model.state_dict(), self.path)
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss = criterion(logits, labels)
            probs = F.softmax(logits, dim=1)
            preds = probs.argmax(1)
            total_loss += loss.item() * imgs.size(0)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)
            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    probs  = np.concatenate(all_probs)
    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    return total_loss / total, correct / total, probs, preds, labels


# =============================================================================
# METRICS
# =============================================================================
def specificity_per_class(cm):
    specs = []
    for i in range(len(cm)):
        tn = cm.sum() - cm[i, :].sum() - cm[:, i].sum() + cm[i, i]
        fp = cm[:, i].sum() - cm[i, i]
        specs.append(tn / (tn + fp + 1e-9))
    return np.array(specs)


def compute_all_metrics(labels, preds, probs, class_names, conf_threshold=0.9):
    report = classification_report(labels, preds, target_names=class_names,
                                   digits=4, output_dict=True)
    cm = confusion_matrix(labels, preds)
    bal_acc = balanced_accuracy_score(labels, preds)
    kappa   = cohen_kappa_score(labels, preds)
    mcc     = matthews_corrcoef(labels, preds)
    specs   = specificity_per_class(cm)
    avg_spec = specs.mean()
    failure_rate = 1 - (labels == preds).mean()

    max_probs = probs.max(axis=1)
    hc_mask   = max_probs >= conf_threshold
    hc_failures = ((preds != labels) & hc_mask).sum()

    return {
        'report'      : report,
        'cm'          : cm,
        'bal_acc'     : bal_acc,
        'kappa'       : kappa,
        'mcc'         : mcc,
        'avg_spec'    : avg_spec,
        'specs'       : specs,
        'failure_rate': failure_rate,
        'hc_failures' : hc_failures,
        'n_hc'        : hc_mask.sum(),
    }


# =============================================================================
# PLOTTING — ALL FIGURES
# =============================================================================
def _save(fig, name):
    path = os.path.join(CFG['output_dir'], name)
    fig.savefig(path, dpi=600, bbox_inches='tight', format='png')
    plt.close(fig)
    print(f"  Saved: {path}")


def plot_confusion_matrix(cm, class_names, title='Confusion Matrix', fname='cm.png'):
    n = len(class_names)
    # Dynamic figure size: minimum 6x5, grows with n
    fig_w = max(6, n * 1.2)
    fig_h = max(5, n * 1.0)
    font_annot = max(8, min(16, int(120 / n)))

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=12)

    tick_marks = np.arange(n)
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels(class_names, rotation=45, ha='right',
                       fontsize=max(7, min(12, int(100 / n))), fontweight='bold')
    ax.set_yticklabels(class_names,
                       fontsize=max(7, min(12, int(100 / n))), fontweight='bold')

    thresh = cm.max() / 2.0
    for i, j in itertools.product(range(n), range(n)):
        ax.text(j, i, f'{cm[i,j]}',
                ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black',
                fontsize=font_annot, fontweight='bold')

    ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.tight_layout()
    _save(fig, fname)


def plot_roc_curves(labels, probs, class_names, fname='roc.png'):
    n = len(class_names)
    fig_w = max(8, n * 0.6 + 5)
    fig, ax = plt.subplots(figsize=(fig_w, 6))

    from sklearn.preprocessing import label_binarize
    lb = label_binarize(labels, classes=list(range(n)))

    colors = plt.cm.tab20(np.linspace(0, 1, n))
    lines = []
    for i, (cname, col) in enumerate(zip(class_names, colors)):
        if n == 2 and i == 0:
            fpr, tpr, _ = roc_curve(lb[:, i], probs[:, i])
        else:
            fpr, tpr, _ = roc_curve(lb[:, i], probs[:, i])
        roc_auc = auc(fpr, tpr)
        l, = ax.plot(fpr, tpr, color=col, lw=1.8)
        lines.append((l, f'{cname} (AUC={roc_auc:.4f})'))

    ax.plot([0,1],[0,1],'k--', lw=1)
    ax.set_xlabel('False Positive Rate', fontweight='bold', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontweight='bold', fontsize=12)
    ax.set_title('ROC Curves', fontweight='bold', fontsize=14)
    ax.tick_params(labelsize=10)

    # Legend OUTSIDE
    leg = ax.legend([l for l,_ in lines], [lbl for _,lbl in lines],
                    loc='upper left', bbox_to_anchor=(1.02, 1),
                    borderaxespad=0, fontsize=9, frameon=True)
    plt.tight_layout()
    _save(fig, fname)


def plot_prc_curves(labels, probs, class_names, fname='prc.png'):
    n = len(class_names)
    from sklearn.preprocessing import label_binarize
    lb = label_binarize(labels, classes=list(range(n)))
    fig_w = max(8, n * 0.6 + 5)
    fig, ax = plt.subplots(figsize=(fig_w, 6))
    colors = plt.cm.tab20(np.linspace(0, 1, n))
    lines = []
    for i, (cname, col) in enumerate(zip(class_names, colors)):
        prec, rec, _ = precision_recall_curve(lb[:, i], probs[:, i])
        ap = average_precision_score(lb[:, i], probs[:, i])
        l, = ax.plot(rec, prec, color=col, lw=1.8)
        lines.append((l, f'{cname} (AP={ap:.4f})'))

    ax.set_xlabel('Recall', fontweight='bold', fontsize=12)
    ax.set_ylabel('Precision', fontweight='bold', fontsize=12)
    ax.set_title('Precision-Recall Curves', fontweight='bold', fontsize=14)
    ax.tick_params(labelsize=10)
    leg = ax.legend([l for l,_ in lines], [lbl for _,lbl in lines],
                    loc='upper left', bbox_to_anchor=(1.02, 1),
                    borderaxespad=0, fontsize=9, frameon=True)
    plt.tight_layout()
    _save(fig, fname)


def plot_training_curves(train_losses, val_losses, train_accs, val_accs, fname='training_curves.png'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(train_losses) + 1)

    ax1.plot(epochs, train_losses, 'b-o', markersize=3, label='Train Loss')
    ax1.plot(epochs, val_losses,   'r-o', markersize=3, label='Val Loss')
    ax1.set_title('Training & Validation Loss', fontweight='bold', fontsize=13)
    ax1.set_xlabel('Epoch', fontweight='bold'); ax1.set_ylabel('Loss', fontweight='bold')
    ax1.legend(fontsize=10); ax1.grid(alpha=0.3)

    ax2.plot(epochs, train_accs, 'b-o', markersize=3, label='Train Acc')
    ax2.plot(epochs, val_accs,   'r-o', markersize=3, label='Val Acc')
    ax2.set_title('Training & Validation Accuracy', fontweight='bold', fontsize=13)
    ax2.set_xlabel('Epoch', fontweight='bold'); ax2.set_ylabel('Accuracy', fontweight='bold')
    ax2.legend(fontsize=10); ax2.grid(alpha=0.3)

    plt.tight_layout()
    _save(fig, fname)


def plot_misclassification_analysis(dataset, labels, preds, probs, class_names, n_show=16, fname='misclassification.png'):
    wrong_idx = np.where(labels != preds)[0]
    if len(wrong_idx) == 0:
        print("  No misclassifications to plot.")
        return
    wrong_idx = wrong_idx[:n_show]
    cols = 4
    rows = max(1, int(np.ceil(len(wrong_idx) / cols)))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    axes = np.array(axes).reshape(-1) if rows * cols > 1 else [axes]

    raw_tf = get_raw_transform()
    for ax_i, idx in enumerate(wrong_idx):
        img_path, _ = dataset.samples[idx]
        img = Image.open(img_path).convert('RGB')
        img_t = raw_tf(img).permute(1, 2, 0).numpy()
        axes[ax_i].imshow(img_t)
        true_lbl = class_names[labels[idx]]
        pred_lbl = class_names[preds[idx]]
        conf     = probs[idx, preds[idx]]
        axes[ax_i].set_title(f'T:{true_lbl}\nP:{pred_lbl} ({conf:.2f})',
                             fontsize=8, fontweight='bold', color='red')
        axes[ax_i].axis('off')

    for ax_j in range(len(wrong_idx), len(axes)):
        axes[ax_j].axis('off')

    fig.suptitle('Misclassification Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    _save(fig, fname)


def plot_confidence_calibration(labels, probs, preds, class_names, n_bins=10, fname='calibration.png'):
    n = len(class_names)
    cols = min(n, 4)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.array(axes).reshape(-1)

    for i, cname in enumerate(class_names):
        binary_labels = (labels == i).astype(int)
        class_probs   = probs[:, i]
        try:
            frac_pos, mean_pred = calibration_curve(binary_labels, class_probs, n_bins=n_bins)
            axes[i].plot(mean_pred, frac_pos, 's-', label='Model', color='steelblue', lw=2)
            axes[i].plot([0,1],[0,1],'k--', label='Perfect', lw=1.5)
            axes[i].set_title(cname, fontweight='bold', fontsize=10)
            axes[i].set_xlabel('Mean Predicted Prob', fontweight='bold', fontsize=9)
            axes[i].set_ylabel('Fraction Positives', fontweight='bold', fontsize=9)
            axes[i].legend(fontsize=8); axes[i].grid(alpha=0.3)
        except Exception:
            axes[i].set_title(f'{cname} (skip)', fontsize=9)

    for j in range(n, len(axes)):
        axes[j].axis('off')

    fig.suptitle('Confidence Calibration Curves', fontsize=14, fontweight='bold')
    plt.tight_layout()
    _save(fig, fname)


def plot_top_confused_pairs(cm, class_names, top_n=10, fname='confused_pairs.png'):
    n = len(class_names)
    pairs = []
    for i in range(n):
        for j in range(n):
            if i != j and cm[i, j] > 0:
                pairs.append((class_names[i], class_names[j], cm[i, j]))
    pairs.sort(key=lambda x: -x[2])
    pairs = pairs[:top_n]
    if not pairs:
        print("  No confused pairs."); return

    labels_pairs = [f'{a} → {b}' for a, b, _ in pairs]
    counts = [c for _, _, c in pairs]

    fig, ax = plt.subplots(figsize=(max(8, len(pairs) * 0.9), 5))
    bars = ax.bar(labels_pairs, counts, color=plt.cm.Reds(np.linspace(0.4, 0.9, len(pairs))))
    ax.set_xticklabels(labels_pairs, rotation=45, ha='right', fontweight='bold', fontsize=10)
    ax.set_ylabel('Count', fontweight='bold', fontsize=11)
    ax.set_title('Top Most Confused Class Pairs', fontweight='bold', fontsize=13)
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(cnt), ha='center', va='bottom', fontweight='bold', fontsize=10)
    plt.tight_layout()
    _save(fig, fname)


def print_metrics_table(metrics, class_names, model_info=None):
    print("\n" + "="*65)
    print("  EVALUATION METRICS SUMMARY")
    print("="*65)
    if model_info:
        for k, v in model_info.items():
            print(f"  {k:<30} {v}")
        print("-"*65)

    report = metrics['report']
    print(f"\n  {'Class':<20} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    print("  " + "-"*62)
    for cn in class_names:
        if cn in report:
            r = report[cn]
            # Per-class accuracy = recall (TP/(TP+FN))
            acc_val = r['recall']
            print(f"  {cn:<20} {acc_val:>10.4f} {r['precision']:>10.4f} {r['recall']:>10.4f} {r['f1-score']:>10.4f}")
    print("  " + "-"*62)
    wa = report.get('weighted avg', {})
    print(f"  {'Weighted Avg':<20} {'-':>10} {wa.get('precision',0):>10.4f} {wa.get('recall',0):>10.4f} {wa.get('f1-score',0):>10.4f}")
    print(f"  {'Overall Accuracy':<20} {report.get('accuracy',0):>10.4f}")

    print(f"\n  Balanced Accuracy : {metrics['bal_acc']:.4f}")
    print(f"  Avg Specificity   : {metrics['avg_spec']:.4f}")
    print(f"  MCC               : {metrics['mcc']:.4f}")
    print(f"  Cohen's Kappa     : {metrics['kappa']:.4f}")
    print(f"  Failure Rate      : {metrics['failure_rate']:.4f}")
    print(f"  High-Conf Failures: {metrics['hc_failures']} / {metrics['n_hc']} (conf≥{CFG['conf_threshold']})")

    spec_str = "  Per-Class Specificity: " + \
               ", ".join([f"{cn}={v:.4f}" for cn, v in zip(class_names, metrics['specs'])])
    print(spec_str)
    print("="*65 + "\n")


# =============================================================================
# XAI — GRAD-CAM + ATTENTION ROLLOUT
# =============================================================================
def get_attention_rollout(model, img_tensor, top_idx):
    """Simple attention rollout approximation using stored top_idx."""
    # Returns a 1D mask over the selected patch positions
    K = top_idx.shape[1]
    attn_map = np.ones(K) / K   # uniform fallback if hooks not set
    return attn_map


def compute_xai(model, dataset, labels, preds, probs, class_names, n_show=8, fname='xai_gradcam.png'):
    if not GRADCAM_AVAILABLE:
        print("  [XAI] pytorch-grad-cam not installed. Skipping Grad-CAM.")
        return

    target_layer = [model.extractor.proj[0]]  # last conv before BN
    cam = GradCAM(model=model, target_layers=target_layer)
    raw_tf = get_raw_transform()
    norm_tf = get_transforms(train=False)

    # Show n_show samples (mix correct + wrong)
    wrong = list(np.where(labels != preds)[0])
    right = list(np.where(labels == preds)[0])
    show_idx = (wrong[:n_show//2] + right[:n_show//2])[:n_show]

    cols = 4
    rows = int(np.ceil(len(show_idx) / cols)) * 2  # raw + cam rows
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 2.5))
    axes = np.array(axes).reshape(-1)

    for i, idx in enumerate(show_idx):
        img_path, _ = dataset.samples[idx]
        raw_img = raw_tf(Image.open(img_path).convert('RGB'))
        inp     = norm_tf(Image.open(img_path).convert('RGB')).unsqueeze(0).to(CFG['device'])

        grayscale_cam = cam(input_tensor=inp)
        rgb_img = raw_img.permute(1,2,0).numpy()
        cam_img = show_cam_on_image(rgb_img, grayscale_cam[0], use_rgb=True)

        row_base = (i // cols) * 2
        col_pos  = i % cols

        ax_raw = axes[row_base * cols + col_pos]
        ax_cam = axes[(row_base + 1) * cols + col_pos]

        ax_raw.imshow(rgb_img)
        ax_raw.set_title(f'T:{class_names[labels[idx]]}', fontsize=7, fontweight='bold')
        ax_raw.axis('off')

        ax_cam.imshow(cam_img)
        correct = '✓' if labels[idx] == preds[idx] else '✗'
        ax_cam.set_title(f'{correct} P:{class_names[preds[idx]]} ({probs[idx,preds[idx]]:.2f})',
                         fontsize=7, fontweight='bold',
                         color='green' if labels[idx]==preds[idx] else 'red')
        ax_cam.axis('off')

    for j in range(len(show_idx) * 2, len(axes)):
        axes[j].axis('off')

    fig.suptitle('Grad-CAM Explanations (CNN Feature Maps)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    _save(fig, fname)
    print("  [XAI] Grad-CAM saved.")


# =============================================================================
# 5-FOLD CROSS VALIDATION
# =============================================================================
def run_kfold(full_dataset, class_names, device):
    print("\n" + "="*65)
    print("  5-FOLD CROSS VALIDATION")
    print("="*65)

    # Focal Loss with class weights computed on full dataset
    class_weights = get_class_weights(full_dataset).to(device)
    criterion     = FocalLoss(alpha=class_weights, gamma=2.0)

    labels_all = [full_dataset.targets[i] for i in range(len(full_dataset))]
    skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True,
                          random_state=CFG['seed'])

    fold_cms     = []
    fold_reports = []
    fold_accs    = []

    for fold, (train_idx, val_idx) in enumerate(
            skf.split(np.zeros(len(labels_all)), labels_all)):
        print(f"\n  --- Fold {fold+1}/{CFG['n_folds']} ---")

        class AugSubset(Dataset):
            def __init__(self, subset, tf):
                self.subset = subset
                self.tf     = tf
            def __len__(self): return len(self.subset)
            def __getitem__(self, i):
                path, _ = self.subset.dataset.samples[
                    self.subset.indices[i]]
                img = Image.open(path).convert('RGB')
                return self.tf(img), self.subset.dataset.targets[
                    self.subset.indices[i]]

        train_subset = Subset(full_dataset, train_idx)
        val_subset   = Subset(full_dataset, val_idx)

        # WeightedRandomSampler per fold (based on fold's training targets)
        fold_tr_targets = [full_dataset.targets[i] for i in train_idx]
        fold_counts     = np.bincount(fold_tr_targets,
                                      minlength=len(class_names))
        fold_class_w    = 1.0 / (fold_counts.astype(float) + 1e-9)
        fold_sample_w   = torch.tensor(
            [fold_class_w[t] for t in fold_tr_targets], dtype=torch.float32)
        fold_sampler    = WeightedRandomSampler(
            fold_sample_w, num_samples=len(train_idx), replacement=True)

        train_loader = DataLoader(
            AugSubset(train_subset, get_transforms(True)),
            batch_size=CFG['batch_size'],
            sampler=fold_sampler,
            num_workers=CFG['num_workers']
        )
        val_loader = DataLoader(
            AugSubset(val_subset, get_transforms(False)),
            batch_size=CFG['batch_size'], shuffle=False,
            num_workers=CFG['num_workers']
        )

        model = LDASN(num_classes=len(class_names)).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'],
                                      weight_decay=CFG['weight_decay'])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG['epochs'])
        es = EarlyStopping(
            CFG['patience'],
            os.path.join(CFG['output_dir'], f'best_fold{fold+1}.pt')
        )

        for epoch in range(CFG['epochs']):
            train_one_epoch(model, train_loader, optimizer, criterion, device)
            _, va, _, _, _ = evaluate(model, val_loader, criterion, device)
            scheduler.step()
            if es.step(va, model):
                print(f"    Early stop @ epoch {epoch+1}")
                break

        model.load_state_dict(torch.load(es.path, map_location=device))
        _, _, probs, preds, true_labels = evaluate(
            model, val_loader, criterion, device)

        cm  = confusion_matrix(true_labels, preds)
        rep = classification_report(true_labels, preds,
                                    target_names=class_names,
                                    digits=4, output_dict=True)
        acc = rep['accuracy']
        fold_cms.append(cm)
        fold_reports.append(rep)
        fold_accs.append(acc)

        print(f"    Fold {fold+1} Accuracy: {acc:.4f}")
        plot_confusion_matrix(cm, class_names,
                              title=f'Confusion Matrix — Fold {fold+1}',
                              fname=f'cm_fold{fold+1}.png')
        print(f"\n  Fold {fold+1} Classification Report:")
        print(classification_report(true_labels, preds,
                                    target_names=class_names, digits=4))

    # --- Overall summary ---
    print("\n" + "="*65)
    print("  5-FOLD OVERALL SUMMARY")
    print("="*65)
    print(f"  Mean Accuracy : {np.mean(fold_accs):.4f}"
          f" ± {np.std(fold_accs):.4f}")

    mean_cm = np.mean(fold_cms, axis=0).astype(int)
    plot_confusion_matrix(mean_cm, class_names,
                          title='Mean Confusion Matrix (5-Fold CV)',
                          fname='cm_5fold_mean.png')

    print("\n  Mean Per-Class Metrics (5-Fold):")
    print(f"  {'Class':<30} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    print("  " + "-"*65)
    for cn in class_names:
        p_vals = [r[cn]['precision'] for r in fold_reports if cn in r]
        r_vals = [r[cn]['recall']    for r in fold_reports if cn in r]
        f_vals = [r[cn]['f1-score']  for r in fold_reports if cn in r]
        if p_vals:
            print(f"  {cn:<30} {np.mean(p_vals):>10.4f}"
                  f" {np.mean(r_vals):>10.4f} {np.mean(f_vals):>10.4f}")

    return fold_cms, fold_reports, fold_accs


# =============================================================================
# MAIN PIPELINE
# =============================================================================
def main():
    device = CFG['device']
    print(f"\n  Device: {device}")

    # ---- Load Dataset ----
    full_dataset_raw = ImageFolder(CFG['data_dir'], transform=get_transforms(False))
    class_names = full_dataset_raw.classes
    n_classes   = len(class_names)
    print(f"  Classes ({n_classes}): {class_names}")
    print(f"  Total samples: {len(full_dataset_raw)}")

    # Train/Val split (80/20) for final model
    all_idx    = np.arange(len(full_dataset_raw))
    all_labels = np.array(full_dataset_raw.targets)
    from sklearn.model_selection import train_test_split
    tr_idx, va_idx = train_test_split(all_idx, test_size=0.2,
                                      stratify=all_labels, random_state=CFG['seed'])

    class TransformSubset(Dataset):
        def __init__(self, dataset, indices, tf):
            self.dataset  = dataset
            self.indices  = indices
            self.tf       = tf
        def __len__(self): return len(self.indices)
        def __getitem__(self, i):
            path, lbl = self.dataset.samples[self.indices[i]]
            img = Image.open(path).convert('RGB')
            return self.tf(img), lbl

    # ---- Class weights & Focal Loss ----
    class_weights = get_class_weights(full_dataset_raw).to(device)
    criterion     = FocalLoss(alpha=class_weights, gamma=2.0)
    print(f"\n  Class weights (inverse freq):")
    for cn, w in zip(class_names, class_weights.cpu().numpy()):
        print(f"    {cn:<30} {w:.4f}")

    # ---- WeightedRandomSampler for training split ----
    tr_sample_w = get_sample_weights(
        type('D', (), {'targets': [full_dataset_raw.targets[i] for i in tr_idx]})()
    )
    sampler = WeightedRandomSampler(
        weights=torch.tensor(tr_sample_w, dtype=torch.float32),
        num_samples=len(tr_idx),
        replacement=True
    )

    train_loader = DataLoader(
        TransformSubset(full_dataset_raw, tr_idx, get_transforms(True)),
        batch_size=CFG['batch_size'],
        sampler=sampler,          # replaces shuffle=True
        num_workers=CFG['num_workers']
    )
    val_loader = DataLoader(
        TransformSubset(full_dataset_raw, va_idx, get_transforms(False)),
        batch_size=CFG['batch_size'], shuffle=False,
        num_workers=CFG['num_workers']
    )

    # ---- Build Model ----
    model = LDASN(num_classes=n_classes).to(device)
    n_params = count_parameters(model)
    flops    = compute_flops(model, device)
    print(f"\n  Trainable Parameters : {n_params:,}")
    print(f"  FLOPs                : {flops}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'],
                                   weight_decay=CFG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'])
    # criterion already defined above (FocalLoss)
    es = EarlyStopping(CFG['patience'], os.path.join(CFG['output_dir'], 'best_model.pt'))

    # ---- Training Loop ----
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    print("\n  Training...")
    for epoch in range(CFG['epochs']):
        tl, ta = train_one_epoch(model, train_loader, optimizer, criterion, device)
        vl, va, _, _, _ = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        train_losses.append(tl); val_losses.append(vl)
        train_accs.append(ta);   val_accs.append(va)
        print(f"  Ep {epoch+1:03d} | TL:{tl:.4f} TA:{ta:.4f} | VL:{vl:.4f} VA:{va:.4f}")
        if es.step(va, model):
            print(f"  Early stopping @ epoch {epoch+1}")
            break

    # ---- Load Best & Final Evaluation ----
    model.load_state_dict(torch.load(es.path, map_location=device))
    inf_time = measure_inference_time(model, device)
    _, _, probs, preds, true_labels = evaluate(model, val_loader, criterion, device)

    metrics = compute_all_metrics(true_labels, preds, probs, class_names, CFG['conf_threshold'])
    model_info = {
        'Trainable Parameters': f'{n_params:,}',
        'FLOPs'               : str(flops),
        'Inference Time (ms)' : f'{inf_time:.2f}',
    }
    print_metrics_table(metrics, class_names, model_info)

    # ---- All Figures ----
    print("\n  Generating figures...")
    plot_training_curves(train_losses, val_losses, train_accs, val_accs)
    plot_confusion_matrix(metrics['cm'], class_names, title='Confusion Matrix', fname='cm_final.png')
    plot_roc_curves(true_labels, probs, class_names)
    plot_prc_curves(true_labels, probs, class_names)

    # Misclassification — need original dataset with .samples
    val_dataset_raw = TransformSubset(full_dataset_raw, va_idx, get_transforms(False))
    # Attach .samples for misclassification & XAI
    val_dataset_raw.samples = [full_dataset_raw.samples[i] for i in va_idx]
    plot_misclassification_analysis(val_dataset_raw, true_labels, preds, probs, class_names)
    plot_confidence_calibration(true_labels, probs, preds, class_names)
    plot_top_confused_pairs(metrics['cm'], class_names)

    # XAI
    compute_xai(model, val_dataset_raw, true_labels, preds, probs, class_names)

    # ---- 5-Fold CV ----
    run_kfold(full_dataset_raw, class_names, device)

    print("\n  All done. Results saved to:", CFG['output_dir'])


if __name__ == '__main__':
    main()


  Device: cuda
  Classes (7): ['Bacterial Blight', 'Curl Virus', 'Healthy Leaf', 'Herbicide Growth Damage', 'Leaf Hopper Jassids', 'Leaf Redding', 'Leaf Variegation']
  Total samples: 2137

  Class weights (inverse freq):
    Bacterial Blight               0.9798
    Curl Virus                     0.5683
    Healthy Leaf                   0.9531
    Herbicide Growth Damage        0.8748
    Leaf Hopper Jassids            1.0886
    Leaf Redding                   0.4238
    Leaf Variegation               2.1116


Unsupported operator aten::gelu encountered 11 time(s)
Unsupported operator aten::sigmoid encountered 2 time(s)
Unsupported operator aten::mul encountered 29 time(s)
Unsupported operator aten::avg_pool2d encountered 1 time(s)
Unsupported operator aten::topk encountered 1 time(s)
Unsupported operator aten::add encountered 10 time(s)
Unsupported operator aten::unfold encountered 2 time(s)
Unsupported operator aten::embedding encountered 1 time(s)
Unsupported operator aten::mean encountered 1 time(s)
Unsupported operator aten::div encountered 5 time(s)
Unsupported operator aten::unflatten encountered 4 time(s)
Unsupported operator aten::softmax encountered 4 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
transf


  Trainable Parameters : 10,552,721
  FLOPs                : 1003288832

  Training...
  Ep 001 | TL:1.0396 TA:0.2756 | VL:0.7319 VA:0.2290
  Ep 002 | TL:0.6819 TA:0.4283 | VL:0.4317 VA:0.4299
  Ep 003 | TL:0.5214 TA:0.5243 | VL:0.3499 VA:0.4790
  Ep 004 | TL:0.4513 TA:0.5705 | VL:0.3983 VA:0.4650
  Ep 005 | TL:0.3797 TA:0.5963 | VL:0.3333 VA:0.5771
  Ep 006 | TL:0.3188 TA:0.6589 | VL:0.2704 VA:0.6075
  Ep 007 | TL:0.3277 TA:0.6390 | VL:0.3065 VA:0.5771
  Ep 008 | TL:0.3097 TA:0.6659 | VL:0.2100 VA:0.6589
  Ep 009 | TL:0.2615 TA:0.7104 | VL:0.2650 VA:0.5514
  Ep 010 | TL:0.2580 TA:0.7068 | VL:0.2853 VA:0.5771
  Ep 011 | TL:0.2230 TA:0.7361 | VL:0.2105 VA:0.6121
  Ep 012 | TL:0.2315 TA:0.7185 | VL:0.1950 VA:0.6706
  Ep 013 | TL:0.2173 TA:0.7367 | VL:0.2065 VA:0.6565
  Ep 014 | TL:0.2062 TA:0.7443 | VL:0.1552 VA:0.7033
  Ep 015 | TL:0.1944 TA:0.7607 | VL:0.2251 VA:0.6145
  Ep 016 | TL:0.1607 TA:0.7712 | VL:0.1779 VA:0.6963
  Ep 017 | TL:0.1564 TA:0.7981 | VL:0.1710 VA:0.7009
  Ep 018 | 